# YouTube AI Agent
## Local Assistance Day Summary_Aug 2025

Code modified by: Arief

### imports

In [2]:
from youtube_transcript_api import YouTubeTranscriptApi
import re
from agents import Agent, function_tool, run_demo_loop
from dotenv import load_dotenv
import asyncio

ModuleNotFoundError: No module named 'youtube_transcript_api'

In [ ]:
# import environment variables from .env file
load_dotenv()

True

### define instructions

In [ ]:
instructions = "You provide help with tasks related to YouTube videos."

### define tool

In [ ]:
@function_tool
def fetch_youtube_transcript(url: str) -> str:
    """
    Extract transcript with timestamps from a YouTube video URL and format it for LLM consumption
    
    Args:
        url (str): YouTube video URL
        
    Returns:
        str: Formatted transcript with timestamps, where each entry is on a new line
             in the format: "[MM:SS] Text"
    """
    # Extract video ID from URL
    video_id_pattern = r'(?:v=|\/)([0-9A-Za-z_-]{11}).*'
    video_id_match = re.search(video_id_pattern, url)
    
    if not video_id_match:
        raise ValueError("Invalid YouTube URL")
    
    video_id = video_id_match.group(1)
    
    try:
        ytt_api = YouTubeTranscriptApi()
        transcript = ytt_api.fetch(video_id)
        
        # Format each entry with timestamp and text
        formatted_entries = []
        for entry in transcript:
            # Convert seconds to MM:SS format
            minutes = int(entry.start // 60)
            seconds = int(entry.start % 60)
            timestamp = f"[{minutes:02d}:{seconds:02d}]"

            formatted_entry = f"{timestamp} {entry.text}"
            formatted_entries.append(formatted_entry)
        
        # Join all entries with newlines
        return "\n".join(formatted_entries)
    
    except Exception as e:
        raise Exception(f"Error fetching transcript: {str(e)}")

### create agent

In [ ]:
agent = Agent(
    name="YouTube Transcript Agent",
    instructions=instructions,
    tools=[fetch_youtube_transcript],
)

### main() function

[Example](https://github.com/ShawhinT/AI-Builders-Bootcamp-6/blob/main/session-4/example_1-youtube_agent.ipynb) with custom CLI

In [ ]:
async def main():
    await run_demo_loop(agent)

In [ ]:
await main()
# what is this video about? https://youtu.be/875v0Ij8_Zw?si=LTIOcfuYLkMHPzZr 

In [ ]:
# # to run in a .py script use
# if __name__ == "__main__":
#     asyncio.run(main())